(iata, gps, anotherone) -> (ident)
Check if there are multiple of () that map -> to same (ident)

american location mapping 

In [1]:
import polars as pl
df = pl.read_csv("/Users/jonahgoode/Documents/PlaneQuery/Code/planequery-data/data/interim/airports.csv")

In [2]:
from airports.airport_lookup import AirportLookup
airport_lookup = AirportLookup()

Loading airports from /Volumes/T2-SSD/planequery/data/raw/ourairports/airports.csv


In [4]:
code_to_airports = {}

for row in df.iter_rows(named=True):
    ident = row["ident"]
    local_code = row["local_code"]
    gps_code = row["gps_code"]
    icao_code = row["icao_code"]
    iata_code = row["iata_code"]
    codes = [ident, local_code, gps_code, icao_code, iata_code]
    airport = airport_lookup.get_Airport_from_airport_ident(row["ident"])
    for code in codes:
        if code is not None and airport is not None:
            code_to_airports.setdefault(code, set()).add(airport)

code_to_airports

{'00AA': {Airport(id=323361, ident='00AA', type=<Airport_Types.SMALL_AIRPORT: 'small_airport'>, name='Aero B Ranch Airport', latitude_deg=38.704022, longitude_deg=-101.473911, elevation_ft=3435, continent='NA', iso_country='US', iso_region='US-KS', municipality='Leoti', scheduled_service='no', icao_code=None, iata_code=None, gps_code='00AA', local_code='00AA', home_link=None, wikipedia_link=None, keywords=None, distance_from_plane_point_in_km=0.0)},
 '00AK': {Airport(id=6524, ident='00AK', type=<Airport_Types.SMALL_AIRPORT: 'small_airport'>, name='Lowell Field', latitude_deg=59.947733, longitude_deg=-151.692524, elevation_ft=450, continent='NA', iso_country='US', iso_region='US-AK', municipality='Anchor Point', scheduled_service='no', icao_code=None, iata_code=None, gps_code='00AK', local_code='00AK', home_link=None, wikipedia_link=None, keywords=None, distance_from_plane_point_in_km=0.0)},
 '00AL': {Airport(id=6525, ident='00AL', type=<Airport_Types.SMALL_AIRPORT: 'small_airport'>, na

In [ ]:
def get_airport_from_code(code, lat, lon, code_priority = ["local_code", "ident", "icao_code", "gps_code"], priortize_faa_airspace = True): # maybe make lat, lon nullabe. 
    if code not in code_to_airports:
        return None
    airports = code_to_airports[code]
    if len(airports) == 1:
        return next(iter(airports))
    if priortize_faa_airspace:
        airports = [airport for airport in airports if airport.is_in_faa_airspace()]
    if len(airports) == 1:
        return next(iter(airports))
    for column in code_priority:
        matches = [
            airport
            for airport in airports
            if getattr(airport, column) == code
        ]

        if len(matches) == 1:
            return matches[0]

        if matches:
            airports = matches
    raise ValueError(f"Multiple airports found for code '{code}' with no clear priority: {airports}")
    if False and lat is not None and lon is not None:
        closest_airport = None
        closest_distance = float('inf')
        for airport in airports:
            distance = airport.distance_to(lat, lon)
            if distance < closest_distance:
                closest_distance = distance
                closest_airport = airport
        return closest_airport

In [7]:
COLUMNS = ["ident", "icao_code", "iata_code", "gps_code", "local_code"]
df.select(COLUMNS)

ident,icao_code,iata_code,gps_code,local_code
str,str,str,str,str
"""00A""",null,null,"""K00A""","""00A"""
"""00AA""",null,null,"""00AA""","""00AA"""
"""00AK""",null,null,"""00AK""","""00AK"""
"""00AL""",null,null,"""00AL""","""00AL"""
"""00AN""",null,null,"""00AN""","""00AN"""
…,…,…,…,…
"""ZYYY""","""ZYYY""",null,"""ZYYY""",null
"""ZZ-0001""",null,null,null,null
"""ZZ-0002""",null,null,null,null


In [8]:
CODE_COLUMNS = ["icao_code", "iata_code", "gps_code", "local_code"]

duplicate_codes = (
    df.select(["ident", *CODE_COLUMNS])
    .unpivot(
        index="ident",
        on=CODE_COLUMNS,
        variable_name="code_type",
        value_name="code",
    )
    .filter(
        pl.col("code").is_not_null()
        & (pl.col("code").str.strip_chars() != "")
    )
    .group_by(["code_type", "code"])
    .agg(
        pl.col("ident").drop_nulls().unique().sort().alias("idents"),
        pl.col("ident").drop_nulls().n_unique().alias("ident_count"),
    )
    .filter(pl.col("ident_count") > 1)
    .sort(["code_type", "code"])
)

In [11]:
df.filter(pl.col("ident") == "KLAX").select(COLUMNS)

ident,icao_code,iata_code,gps_code,local_code
str,str,str,str,str
"""KLAX""","""KLAX""","""LAX""","""KLAX""","""LAX"""


In [9]:
duplicate_codes

code_type,code,idents,ident_count
str,str,list[str],u32
"""local_code""","""14IL""","[""74IL"", ""US-4779""]",2
"""local_code""","""17XA""","[""US-12763"", ""US-4407""]",2
"""local_code""","""19CA""","[""US-0062"", ""US-3057""]",2
"""local_code""","""20NJ""","[""20NJ"", ""US-20NJ""]",2
"""local_code""","""23JY""","[""23JY"", ""US-4487""]",2
…,…,…,…
"""local_code""","""YBL""","[""AYXY"", ""CYBL""]",2
"""local_code""","""YEH""","[""PG-0072"", ""YEH""]",2
"""local_code""","""YOP""","[""AR-0447"", ""SK-133""]",2


In [14]:
duplicate_codes.filter(pl.col("code") == "LAX")

code_type,code,idents,ident_count
str,str,list[str],u32
"""local_code""","""LAX""","[""KLAX"", ""MX-1767""]",2


check if there in th the USA and then we can do a lot of mappings